# LCEL, Runnable, RAG, 라우팅

- OpenAI 예제: `OPENAI_API_KEY` 환경 변수 설정 필요 (API 사용 요금 발생 가능)
- Ollama 예제: 로컬 Ollama 서버를 실행하고 `ollama pull exaone3.5`로 모델 설치 필요
- LLM이 필요 없는 Runnable 예제는 API 키 없이도 실행 가능합니다.


> **한눈에 보기**
> - **LCEL** `A | B | C`: 앞 단계의 출력이 다음 단계의 입력. 모든 부품은 `invoke`/`batch`/`stream`을 가진 **Runnable**
> - `RunnableLambda`(함수 → Runnable), `RunnableParallel`(여러 값 동시에), `RunnablePassthrough.assign`(입력 유지 + 키 추가), `RunnableBranch`(조건 분기)
> - fake retriever로 RAG 체인 구조 보기, `RunnableLambda`가 체인을 반환하는 동적 라우팅
> - **실행 결과의 교훈**: 모델은 "LCEL"을 모르고 매번 다른 약자를 지어냈다. context를 넣은 RAG 체인만 맞게 답했다
> - 정리 문서: [LCEL과Runnable.md](LCEL과Runnable.md) · 슬라이드 이미지는 자리만 표시, 📝 정리 셀은 실행 결과를 보고 직접 정리

In [1]:
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
llm_openai = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
llm_exaone = ChatOllama(model="exaone3.5", temperature=0.7)

### `StrOutputParser`로 출력 파싱

모델 호출 결과는 기본적으로 `AIMessage` 객체입니다. \
매번 `.content`를 꺼내 쓰는 대신, `StrOutputParser`를 체인 마지막에 붙이면 **바로 문자열**을 받을 수 있습니다.\
지금은 단독으로 호출해보고, 다음 절(LCEL 체인)에서\ `|` 연산자로 자연스럽게 연결하는 방법을 학습합다.

In [2]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

response = llm_openai.invoke("파이썬과 자바의 가장 큰 차이를 한 문장으로 말해줘")
print("파싱 전 타입:", type(response))          # AIMessage

parsed = parser.invoke(response)
print("파싱 후 타입:", type(parsed))            # TextAccessor(str)
print("[gpt-4o-mini]", parsed)

파싱 전 타입: <class 'langchain_core.messages.ai.AIMessage'>
파싱 후 타입: <class 'langchain_core.messages.base.TextAccessor'>
[gpt-4o-mini] 파이썬은 동적 타이핑을 지원하여 코드 작성이 간편하고 빠르며, 자바는 정적 타이핑을 통해 더 강력한 타입 검사를 제공하여 대규모 애플리케이션 개발에 적합하다.


**📝 정리** — `StrOutputParser` 결과 타입이 `TextAccessor`로 나왔다. `str`을 상속한 클래스라 문자열로 써도 된다.

In [3]:
response = llm_exaone.invoke("파이썬과 자바의 가장 큰 차이를 한 문장으로 말해줘")
parsed = parser.invoke(response)  # parser는 모델과 무관하게 재사용 가능
print("[exaone3.5]", parsed)

[exaone3.5] 파이썬은 간결한 문법으로 빠른 개발과 데이터 과학에 강점을 둔 반면, 자바는 객체 지향 프로그래밍에 중점을 두고 안정성과 확장성을 강조합니다.


---

### LanChain에서 사용되는 주요 객체의 입력 데이터와 출력 데이터

| 단계 | 입력 예시 | 출력 예시 |
|---|---|---|
| `ChatPromptTemplate` | `dict` | `ChatPromptValue` |
| `ChatOpenAI` / `ChatOllama` | `PromptValue` 또는 메시지 | `AIMessage` |
| `StrOutputParser` | `AIMessage` | `TextAccessor(str)` |
| `JsonOutputParser` | `AIMessage` | `dict` |
| `RunnableParallel` | 보통 공통 입력 | 키별 결과를 담은 `dict` |

LCEL 의 `파이프기호(|)`는 **앞 단계 출력과 다음 단계 입력을 연결**합니다.


### LangChain LCEL 정리

**LCEL(LangChain Expression Language)** 

```python
chain = A | B | C
```

위 코드는 다음 의미한다.

```text
입력 → A → B → C → 출력
```

즉, LCEL은 LangChain의 여러 부품을 **파이프라인처럼 연결하는 문법**이다.\
**chain = summary_preprocess_chain | summary_prompt | model | StrOutputParser()**

[ 장점 ]

- 깔끔한 코드
- 알아서 지원하는 스트리밍 (Streaming)
- 공통 실행 인터페이스: `invoke`(단일), `ainvoke`(비동기), `batch`(여러 입력), `stream`(출력 청크)\
 
[ LCEL의 핵심 부품들 ]

- RunnablePassthrough: 데이터를 가공하지 않고 그대로 통과시키거나, assign()을 통해 새로운 값을 추가할 때 사용(전처리 단계).
- RunnableLambda: 사용자가 만든 일반 Python 함수를 LCEL 파이프라인 중간에 쏙 끼워 넣을 수 있게 래핑
- RunnableBranch: 상황에 따라 이쪽 체인이나 저쪽 체인으로 흐름을 바꾸는 조건문 역할을 수행

---

### 1. LCEL 이해( LLM 없이 )

LCEL의 핵심은 **Runnable**이며 **`Runnable`은 LangChain에서 실행 가능한 작은 블록**입니다.  
일반 Python 함수도 `RunnableLambda`로 감싸면 LCEL 체인에 넣을 수 있습니다.

In [4]:
from langchain_core.runnables import RunnableLambda

strip_text = RunnableLambda(lambda x: x.strip())
to_lower = RunnableLambda(lambda x: x.lower())
add_prefix = RunnableLambda(lambda x: f"정리된 입력: {x}")

chain = strip_text | to_lower | add_prefix
print(type(chain))

result = chain.invoke("   HELLO LCEL     ")
print("|"+result+"|")

<class 'langchain_core.runnables.base.RunnableSequence'>
|정리된 입력: hello lcel|


**📝 정리** — `|`로 이은 결과는 `RunnableSequence`다. LLM 없이도 LCEL의 흐름(앞 출력 → 다음 입력)을 확인할 수 있다.

위 코드는 내부적으로 다음 순서로 실행된다.

```text
"   HELLO LCEL     "
→ strip_text
→ to_lower
→ add_prefix
→ "정리된 입력: hello lcel"
```

`|`는 앞 단계의 출력을 다음 단계의 입력으로 넘긴다.

### 2. `invoke`, `batch`, `stream` 

LCEL 체인은 보통 다음의 실행 인터페이스중 하나를 사용한다.

- `invoke()` : 입력 하나 실행
- `batch()` : 입력 여러 개 실행
- `stream()` : 결과를 조금씩 스트리밍

In [5]:
# invoke: 하나 실행
print(chain.invoke("   LangChain   "))

정리된 입력: langchain


In [6]:
# batch: 여러 개 실행
inputs = [
    "   HELLO   ",
    "   LCEL   ",
    "   LangChain   ",
]
results = chain.batch(inputs)
print(type(results))
for r in results:
    print(r)

<class 'list'>
정리된 입력: hello
정리된 입력: lcel
정리된 입력: langchain


In [7]:
# stream: 가능한 경우 결과를 조각 단위로 받을 수 있다.
# 지금 체인은 문자열 후처리 체인이라 실질적으로는 한 번에 나온다.
for chunk in chain.stream("   STREAM TEST"):
    print(chunk, flush=True)

정리된 입력: stream test


### 3. Prompt → Model → Parser 기본 구조

LCEL에서 가장 흔한 패턴은 다음 구조이다.

```python
chain = prompt | model | parser
```

의미는 다음과 같다.

```text
사용자 입력 dict
→ PromptTemplate이 프롬프트 생성
→ ChatModel이 답변 생성
→ OutputParser가 문자열 또는 원하는 형식으로 변환
```

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

In [8]:
import os

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 초보자에게 쉽게 설명하는 AI 튜터야."),
    ("human", "{topic}에 대해 5줄 이내로 쉽게 설명해줘."),
])

parser = StrOutputParser()

if os.getenv("OPENAI_API_KEY"):
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    basic_llm_chain = prompt | model | parser

    answer = basic_llm_chain.invoke({"topic": "LangChain LCEL"})
    print(answer)
else:
    print("OPENAI_API_KEY가 없어 LLM 호출 예제는 건너뜁니다.")

LangChain LCEL은 언어 모델을 사용하여 다양한 작업을 수행할 수 있도록 돕는 프레임워크입니다. LCEL은 "Language Chain Execution Language"의 약자로, 언어 모델의 기능을 연결하고 조합하여 복잡한 작업을 쉽게 처리할 수 있게 해줍니다. 이를 통해 개발자들은 자연어 처리(NLP) 애플리케이션을 더 효율적으로 만들 수 있습니다. 간단히 말해, LCEL은 언어 모델을 활용한 작업을 체계적으로 관리하고 실행하는 도구입니다.


**📝 정리** — **환각**: gpt-4o-mini가 LCEL을 "Language Chain Execution Language의 약자"라고 지어냈다. 정답은 LangChain Expression Language다. 모델이 잘 모르는 최신 용어는 그럴듯하게 만들어 낸다.

In [9]:
model = ChatOllama(
    model="exaone3.5",
    temperature=0,
)
basic_chain = prompt | model | parser
print(type(basic_chain))
answer = basic_chain.invoke({"topic": "LangChain LCEL"})
print(answer)

<class 'langchain_core.runnables.base.RunnableSequence'>
LangChain LCEL은 **대화형 AI 모델의 실제 성능을 측정하는 방법**이야. 

* **실제 사용 환경**에서 얼마나 자연스럽고 유용한지 평가해줘.
* **단순히 점수만으로는 알 수 없는** 실용적인 강점을 보여줘.
* 예를 들어, 챗봇이 얼마나 잘 문제를 해결하는지 측정할 수 있어.

쉽게 말해, **AI가 얼마나 똑똑한지 실제로 보여주는 척도**라고 생각하면 돼!


**📝 정리** — EXAONE은 "대화형 AI 모델의 실제 성능을 측정하는 방법"이라고 답했다. 역시 틀렸다.

In [10]:
model = ChatOllama(
    model="llama3.1",
    temperature=0,
)
basic_chain = prompt | model | parser
print(type(basic_chain))
answer = basic_chain.invoke({"topic": "LangChain LCEL"})
print(answer)

<class 'langchain_core.runnables.base.RunnableSequence'>
LangChain LCEL은 언어 모델을 사용하여 대화나 텍스트를 생성하는 AI입니다. 

LCEL은 LangChain의 Core Library로, 언어 모델을 사용하여 다양한 텍스트 생성과 처리를 지원합니다. 

사용자는 LCEL을 통해 자연어 처리, 텍스트 생성, 대화 시스템 등 다양한 기능을 구현할 수 있습니다. 

LCEL은 LangChain의 Core Library로, 언어 모델을 사용하여 다양한 텍스트 생성과 처리를 지원합니다. 

사용자는 LCEL을 통해 자연어 처리, 텍스트 생성, 대화 시스템 등 다양한 기능을 구현할 수 있습니다.


**📝 정리** — Llama 3.1은 "LangChain의 Core Library"라고 답하고 같은 문장을 반복했다. 세 모델 모두 LCEL을 몰랐다.

In [11]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

class Answer(BaseModel):
    topic: str = Field(description="질문 주제")
    answer: str = Field(description="설명")

json_parser = JsonOutputParser(pydantic_object=Answer)
json_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 출력 형식을 준수하고 JSON 객체만 출력하세요.\n{format_instructions}"),
    ("human", "{topic}을 5줄 이내로 설명해줘."),
])
json_chain = json_prompt | llm_openai | json_parser

json_result = json_chain.invoke({
    "topic": "LangChain LCEL",
    "format_instructions": json_parser.get_format_instructions(),
})
print(type(json_result))  # dict (Pydantic 모델 인스턴스가 아님)
print(json_result)


<class 'dict'>
{'topic': 'LangChain LCEL', 'answer': 'LangChain LCEL은 대화형 AI 모델을 구축하기 위한 프레임워크로, 다양한 데이터 소스를 활용하여 자연어 처리(NLP) 작업을 효율적으로 수행할 수 있게 도와줍니다. 사용자는 언어 모델과 데이터 소스 간의 연결을 쉽게 설정할 수 있으며, 이를 통해 대화형 애플리케이션을 빠르게 개발할 수 있습니다. LCEL은 모듈화된 구조를 가지고 있어 사용자가 필요에 따라 기능을 확장하거나 커스터마이즈할 수 있습니다.'}


In [12]:
# JSON 파서 출력(dict)을 문자열 함수에 바로 전달하면 타입 오류가 납니다.
# 따라서 문자열 출력 체인을 별도로 만들고 해당 체인에 후처리를 연결합니다.
def clean_output(text: str) -> str:
    while "\n\n\n" in text:
        text = text.replace("\n\n\n", "\n\n")
    return text.strip()

text_prompt = ChatPromptTemplate.from_template("{topic}을 초보자에게 쉽게 설명해줘.")
cleaned_chain = text_prompt | llm_openai | StrOutputParser() | RunnableLambda(clean_output)
print(cleaned_chain.invoke({"topic": "LCEL 후처리"}))


LCEL 후처리(후처리 기술)는 "Low-Carbon Energy Laboratory"의 약자로, 탄소 배출을 줄이기 위해 에너지 생산 및 소비 과정에서 발생하는 이산화탄소(CO2)를 효과적으로 처리하고 관리하는 방법을 의미합니다. 초보자를 위해 쉽게 설명하자면:

1. **탄소 배출 이해하기**: 우리가 자동차를 타거나 전기를 사용할 때, 이산화탄소가 발생합니다. 이산화탄소는 지구 온난화의 주범이기 때문에 줄이는 것이 중요합니다.

2. **후처리의 필요성**: 에너지를 생산하는 과정에서 이산화탄소가 발생하지만, 이를 그냥 대기 중으로 방출하면 환경에 좋지 않습니다. 그래서 이산화탄소를 포집하거나 변환하여 다른 형태로 만드는 기술이 필요합니다.

3. **후처리 방법**:
   - **포집**: 이산화탄소를 에너지를 생산하는 과정에서 직접 포집하여 저장하는 방법입니다.
   - **변환**: 포집한 이산화탄소를 다른 유용한 물질로 변환하여 활용하는 방법입니다. 예를 들어, 이산화탄소를 연료나 화학물질로 바꿀 수 있습니다.

4. **결과**: 이러한 후처리 기술을 통해 대기 중의 이산화탄소를 줄이고, 환경을 보호하며, 지속 가능한 에너지 사용을 촉진할 수 있습니다.

이렇게 LCEL 후처리는 우리가 에너지를 사용할 때 발생하는 문제를 해결하기 위한 중요한 기술입니다.


**📝 정리** — 같은 질문인데 이번엔 "Low-Carbon Energy Laboratory"(탄소 포집 기술)라고 답했다. 실행할 때마다 다른 답을 지어낸다는 것을 확인했다.

- `json_chain`의 입력은 `topic`, `format_instructions`가 들어 있는 `dict`이고, `JsonOutputParser`의 출력도 Python `dict`입니다.  - - `cleaned_chain`의 입력은 `topic`이 들어 있는 `dict`이고, 후처리 입력은 `StrOutputParser()`가 반환한 문자열입니다.
- `JsonOutputParser(pydantic_object=...)`는 JSON 결과를 사전으로 파싱합니다. 
- **Pydantic 객체 반환과 검증을 함께 실습하려면 `PydanticOutputParser`를 사용**하거나 Chat 모델의 `with_structured_output()`을 고려하세요. 

### 4. 입력 shape 실수 확인하기

LCEL에서 제일 많이 헷갈리는 부분은 문법보다 **입력과 출력의 모양**이다.

아래 prompt는 `{name}`이라는 변수를 필요로 한다.

In [13]:
name_prompt = ChatPromptTemplate.from_template("안녕하세요, {name}님! 오늘 배울 주제는 LCEL입니다.")

# 올바른 입력
print(name_prompt.invoke({"name": "둘리"}))

messages=[HumanMessage(content='안녕하세요, 둘리님! 오늘 배울 주제는 LCEL입니다.', additional_kwargs={}, response_metadata={})]


```python
prompt = ChatPromptTemplate.from_template("안녕 {name}")
```
이 prompt는 아래 입력을 기대한다.
```python
{"name": "둘리"}
```
LCEL 체인을 만들 때는 각 단계의 입력과 출력이 서로 맞는지 항상 확인해야 한다.

### 5. Dict / RunnableParallel: 여러 값을 동시에 만들기

LCEL에서는 여러 Runnable을 병렬로 실행해서 dict 형태로 합칠 수 있다.

예를 들어 같은 입력 문자열에서

- 원본 문자열
- 대문자 문자열
- 문자열 길이

를 동시에 만들 수 있다.

In [14]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

parallel_chain = RunnableParallel({
    "original": RunnablePassthrough(),
    "upper": RunnableLambda(lambda x: x.upper()),
    "length": RunnableLambda(lambda x: len(x)),
})

parallel_result = parallel_chain.invoke("  hello lcel  ")
parallel_result

{'original': '  hello lcel  ', 'upper': '  HELLO LCEL  ', 'length': 14}

`RunnablePassthrough()`는 입력을 그대로 넘겨주는 Runnable 로서
RAG 예제에서 아래와 같은 구조를 많이 보게 되는데
```python
{
    "context": retriever,
    "question": RunnablePassthrough()
}
```
의미는 다음과 같다.
```text
같은 입력 질문을 받아서
- context는 retriever로 만들고
- question은 원본 질문을 그대로 유지한다
```

**RAG의 범위 구분:** 여기서는 이해를 돕기 위해 Python 리스트를 검색하는 `fake_retriever`를 사용합니다. 실제 RAG는 문서 수집·청킹·임베딩·벡터 DB/검색기·검색 결과 평가 등이 추가됩니다. 

### 6. Fake Retriever로 RAG 구조 이해하기

이번 단계에서는 실제 벡터 DB 없이, RAG 파이프라인의 구조만 가볍게 흉내 내어 테스트해 본다.

RAG의 기본 흐름은 다음과 같다.

```text
질문
→ 관련 문서 검색
→ 검색 문서를 context에 넣음
→ 질문과 context를 함께 LLM에 전달
→ 답변 생성
```

In [15]:
docs = [
    "LCEL은 LangChain 컴포넌트를 파이프처럼 연결하는 문법이다.",
    "Runnable은 invoke, batch, stream 같은 공통 실행 인터페이스를 가진다.",
    "RAG는 검색된 문서를 context로 넣고 질문에 답하는 방식이다.",
    "RunnablePassthrough는 입력을 그대로 넘기거나 assign으로 값을 추가할 때 유용하다.",
]

# 학습용 키워드 검색: 실제 벡터 검색 / 의미 검색과는 다릅니다.
def fake_retriever(question: str):
    keywords = ["LCEL", "Runnable", "RAG", "context", "stream", "batch", "invoke"]
    question_terms = [k.lower() for k in keywords if k.lower() in question.lower()]
    matched = [doc for doc in docs if any(k in doc.lower() for k in question_terms)]
    return matched or docs[:1]

def augmanted_docs(retrieved_docs):
    return "\n".join(f"- {doc}" for doc in retrieved_docs)

retriever_chain = RunnableLambda(fake_retriever) | RunnableLambda(augmanted_docs)
rag_input_chain = RunnableParallel({
    "context": retriever_chain,
    "question": RunnablePassthrough(),
})
print(rag_input_chain.invoke("LCEL과 Runnable은 무슨 관계야?"))


{'context': '- LCEL은 LangChain 컴포넌트를 파이프처럼 연결하는 문법이다.\n- Runnable은 invoke, batch, stream 같은 공통 실행 인터페이스를 가진다.\n- RunnablePassthrough는 입력을 그대로 넘기거나 assign으로 값을 추가할 때 유용하다.', 'question': 'LCEL과 Runnable은 무슨 관계야?'}


질문 하나를 넣었는데, 아래 형태의 dict가 만들어졌습니다.

```python
{
    "context": "검색된 문서들...",
    "question": "원본 질문..."
}
```
이제 이 dict(전처리 체인)를 prompt에 넣으면 RAG 체인이 된다.

In [16]:
rag_prompt = ChatPromptTemplate.from_template("""
다음 context만 참고해서 질문에 답해줘.

[context]
{context}

[question]
{question}
""")

if os.getenv("OPENAI_API_KEY"):
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    rag_chain = rag_input_chain | rag_prompt | model | StrOutputParser()

    answer = rag_chain.invoke("LCEL과 Runnable은 무슨 관계야?")
    print(answer)
else:
    print("OPENAI_API_KEY가 없어 RAG LLM 호출 예제는 건너뜁니다.")
    print("대신 prompt에 들어갈 입력만 확인합니다:\n")
    print(rag_input_chain.invoke("LCEL과 Runnable은 무슨 관계야?"))

LCEL은 LangChain 컴포넌트를 파이프처럼 연결하는 문법으로, Runnable은 이러한 컴포넌트들이 공통적으로 가지는 실행 인터페이스를 제공하는 요소입니다. 즉, LCEL을 사용하여 Runnable을 연결하고 조합함으로써 다양한 작업을 수행할 수 있습니다. 따라서 LCEL은 Runnable을 활용하는 방식 중 하나라고 할 수 있습니다.


**📝 정리** — 같은 gpt-4o-mini인데 context(LCEL 설명 문서)를 넣어 주니 정확히 답했다. 모델이 모를 수 있는 내용은 **근거를 넣고 그 안에서만 답하게** 하는 것이 RAG의 핵심이다.

RAG에서 가장 자주 쓰는 LCEL 패턴은 결국 아래입니다.

```python
rag_chain = (
    {
        "context": retriever | augmented_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | parser
)
```

이 패턴만 익숙해져도 LangChain RAG 코드의 상당 부분을 파악할 수 있다.

### 7. RunnablePassthrough.assign: 기존 입력에 값 추가하기

`assign`은 기존 dict 입력에 새로운 key를 추가할 때 사용한다.

예를 들어 문장과 함께 문장 길이를 계산해서 넣어보겠습니다.

In [17]:
assign_chain = RunnablePassthrough.assign(
    length=lambda x: len(x["text"]),
    preview=lambda x: x["text"][:10] + "...",
)

assign_result = assign_chain.invoke({
    "text": "LangChain LCEL은 체인을 조립하기 쉽게 해준다."
})

assign_result

{'text': 'LangChain LCEL은 체인을 조립하기 쉽게 해준다.',
 'length': 32,
 'preview': 'LangChain ...'}

**📝 정리** — `assign()`은 원래 `text` 키를 유지하고 `length`, `preview`를 추가했다. 길이 32는 공백과 한글을 모두 한 글자로 센 `len()` 결과다.

초기 입력은 `{"text": "..."}` 형태입니다. `assign()`은 원래 키를 유지하고 `length`, `preview`를 추가합니다. **문자열 길이는 공백과 한글 문자까지 포함한 Python `len()` 결과**이므로, 아래 수치는 직접 실행해서 확인하세요.


In [18]:
analysis_prompt = ChatPromptTemplate.from_template("""
다음 문장을 분석해줘.

문장: {text}
문장 길이: {length}
미리보기: {preview}

간단히 평가해줘.
""")

if os.getenv("OPENAI_API_KEY"):
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    analysis_chain = assign_chain | analysis_prompt | model | StrOutputParser()

    result = analysis_chain.invoke({
        "text": "LangChain LCEL은 체인을 조립하기 쉽게 해준다."
    })

    print(result)
else:
    print("OPENAI_API_KEY가 없어 LLM 호출 예제는 건너뜁니다.")
    print(assign_chain.invoke({"text": "LangChain LCEL은 체인을 조립하기 쉽게 해준다."}))

문장 "LangChain LCEL은 체인을 조립하기 쉽게 해준다."는 다음과 같은 요소로 분석할 수 있습니다.

1. **주어**: "LangChain LCEL" - 특정 기술이나 도구를 지칭.
2. **서술어**: "해준다" - 주어가 하는 행동을 나타냄.
3. **목적어**: "체인을 조립하기 쉽게" - 주어의 행동이 미치는 대상을 설명.

**문장 길이**: 32자는 비교적 짧고 간결한 문장으로, 핵심 내용을 명확하게 전달하고 있습니다.

**미리보기**: "LangChain ..."으로 시작하는 문장은 독자가 이 기술에 대해 더 알고 싶어할 수 있도록 호기심을 유발합니다.

**평가**: 이 문장은 명확하고 직관적이며, 특정 기술의 기능을 간단히 설명하고 있어 이해하기 쉽습니다. 추가적인 정보나 맥락이 제공된다면 더욱 풍부한 내용이 될 수 있을 것입니다.


### 8. RunnableBranch: 조건에 따라 다른 체인 실행하기

질문 유형에 따라 다른 처리 흐름을 수행할 수도 있다.

예를 들어:

```text
코드 질문 → 코드 리뷰어 스타일
클라우드 질문 → 클라우드 엔지니어 스타일
그 외 질문 → 일반 설명 스타일
```

처럼 나눌 수 있다.

- 호출 방식
```
RunnableBranch(
    (condition_1, runnable_1),
    (condition_2, runnable_2),
    # ... 원하는 만큼 (조건, Runnable) 쌍을 추가
    default_runnable # 마지막 인자는 무조건 else 역할
)
```

In [19]:
from langchain_core.runnables import RunnableBranch, RunnableLambda

code_chain = RunnableLambda(lambda x: f"[코드 리뷰어 모드] 질문: {x['question']}")
cloud_chain = RunnableLambda(lambda x: f"[클라우드 엔지니어 모드] 질문: {x['question']}")
general_chain = RunnableLambda(lambda x: f"[일반 설명 모드] 질문: {x['question']}")

router_chain = RunnableBranch(
    (lambda x: "코드" in x["question"] or "파이썬" in x["question"], code_chain),
    (lambda x: "쿠버네티스" in x["question"] or "클라우드" in x["question"], cloud_chain),
    general_chain,
)

questions = [
    {"question": "파이썬 코드에서 에러가 나."},
    {"question": "쿠버네티스 Pod가 Pending이야."},
    {"question": "LCEL을 쉽게 설명해줘."},
]

for q in questions:
    print(router_chain.invoke(q))

[코드 리뷰어 모드] 질문: 파이썬 코드에서 에러가 나.
[클라우드 엔지니어 모드] 질문: 쿠버네티스 Pod가 Pending이야.
[일반 설명 모드] 질문: LCEL을 쉽게 설명해줘.


**📝 정리** — LLM 없이 문자열만 돌려주는 분기로 먼저 동작을 확인했다. 조건은 위에서부터 검사하고 마지막 인자가 else다.

LLM을 붙이면 각 분기마다 다른 프롬프트를 사용할 수 있다.

In [20]:
if os.getenv("OPENAI_API_KEY"):
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    parser = StrOutputParser()

    code_llm_chain = (
        ChatPromptTemplate.from_template(
            "너는 코드 리뷰어야. 다음 질문에 코드 중심으로 답해줘:\n{question}"
        )
        | model
        | parser
    )

    cloud_llm_chain = (
        ChatPromptTemplate.from_template(
            "너는 클라우드 엔지니어야. 다음 질문에 인프라 관점으로 답해줘:\n{question}"
        )
        | model
        | parser
    )

    general_llm_chain = (
        ChatPromptTemplate.from_template(
            "다음 질문에 초보자도 이해하기 쉽게 답해줘:\n{question}"
        )
        | model
        | parser
    )

    llm_router_chain = RunnableBranch(
        (lambda x: "코드" in x["question"] or "파이썬" in x["question"], code_llm_chain),
        (lambda x: "쿠버네티스" in x["question"] or "클라우드" in x["question"], cloud_llm_chain),
        general_llm_chain,
    )

    print(llm_router_chain.invoke({
        "question": "쿠버네티스 Pod가 계속 Pending일 때 어디를 봐야 해?"
    }))
else:
    print("OPENAI_API_KEY가 없어 LLM 라우터 예제는 건너뜁니다.")

쿠버네티스에서 Pod가 Pending 상태에 있는 경우, 여러 가지 원인이 있을 수 있습니다. 인프라 관점에서 확인해야 할 주요 사항은 다음과 같습니다:

1. **리소스 요청 및 할당**:
   - Pod가 요청한 CPU 및 메모리 리소스가 클러스터의 노드에서 사용 가능한지 확인합니다. `kubectl describe pod <pod-name>` 명령어를 사용하여 Pod의 리소스 요청을 확인하고, 클러스터의 노드에서 해당 리소스가 충분히 있는지 점검합니다.

2. **노드의 상태**:
   - 클러스터의 노드가 정상적으로 작동하고 있는지 확인합니다. `kubectl get nodes` 명령어로 노드의 상태를 확인하고, Ready 상태인지 확인합니다. 노드가 NotReady 상태라면, 해당 노드에서 Pod를 스케줄할 수 없습니다.

3. **스케줄러의 문제**:
   - 쿠버네티스 스케줄러가 Pod를 적절한 노드에 할당하지 못하는 경우도 있습니다. 스케줄러의 로그를 확인하여 오류 메시지가 있는지 확인합니다.

4. **Taints와 Tolerations**:
   - 노드에 taint가 설정되어 있는 경우, Pod가 해당 노드에 스케줄되기 위해서는 적절한 toleration이 필요합니다. Pod의 정의에서 toleration이 설정되어 있는지 확인합니다.

5. **Affinity 및 Anti-affinity 규칙**:
   - Pod가 특정 노드에 스케줄되도록 설정된 affinity 규칙이나, 특정 노드에 스케줄되지 않도록 설정된 anti-affinity 규칙이 있는지 확인합니다. 이러한 규칙이 Pod의 스케줄링을 방해할 수 있습니다.

6. **Persistent Volume Claims (PVC)**:
   - Pod가 Persistent Volume을 사용하려고 하는 경우, 해당 PVC가 바인딩되지 않았거나 사용할 수 있는 PV가 없는지 확인합니다. PVC의 상태를 확인하여 바인딩 상태를 점검합니다.

7. **네트워크 정책**:
   - 네트워크 정책

- RunnableBranch는if-else 조건문(분기문)을 LCEL 체인 형태로 만들어 놓은 컴포넌트이다.
- **A 조건일 때는 A 체인으로, B 조건일 때는 B 체인으로**

**스트리밍 주의:** `.stream()`을 호출해도 모든 Runnable이 글자/토큰 단위 출력을 제공하는 것은 아닙니다. `StrOutputParser()`와 채팅 모델을 잇는 파이프라인은 점진적 출력이 가능하지만, 일부 후처리나 외부 모델 설정에 따라 청크가 합쳐지거나 지연될 수 있습니다.


### 9. 스트리밍 예제

LLM 체인에서 `stream()`을 사용하면 답변을 조각 단위로 받을 수 있다.

터미널이나 웹앱에서 실시간으로 답변이 생성되는 것처럼 보여주고 싶을 때 유용하다.

In [21]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

stream_chain = (
    ChatPromptTemplate.from_template("{topic}에 대해 아주 쉽게 설명해줘.")
    | model
    | StrOutputParser()
)

In [22]:

for chunk in stream_chain.stream({"topic": "LCEL stream"}):
    print(chunk, end="@", flush=True)

@LC@EL@(stream@)@은@ "@Low@-C@ost@ Energy@ Loss@"@의@ 약@자로@,@ 에@너지@ 손@실@을@ 최소@화@하기@ 위한@ 저@비@용@의@ 방법@이나@ 기술@을@ 의미@합니다@.@ 주@로@ 전@력@ 시스템@이나@ 에@너지@ 관리@ 분야@에서@ 사용@되@며@,@ 에@너@지를@ 효@율@적으로@ 사용@하고@ 낭@비@를@ 줄@이는@ 데@ 초@점을@ 맞@추@고@ 있습니다@.

@간@단@히@ 말@해@,@ LC@EL@ stream@은@ 에@너@지를@ 더@ 효@율@적으로@ 사용@하고@,@ 비용@을@ 절@감@하며@,@ 환경@에@ 미@치는@ 영향을@ 줄@이@기@ 위한@ 전략@이나@ 방법@을@ 나타@냅니다@.@ 예@를@ 들어@,@ 에@너지@ 절@약@형@ 기@기를@ 사용@하거나@,@ 에@너지@ 소비@ 패@턴@을@ 분석@하여@ 불@필@요@한@ 낭@비@를@ 줄@이는@ 것@ 등이@ 포함@될@ 수@ 있습니다@.@ 

@이@러한@ 접근@은@ 기업@이나@ 가@정@에서@ 에@너지@ 비용@을@ 줄@이고@,@ 지속@ 가능한@ 에@너지@ 사용@을@ 촉@진@하는@ 데@ 도움@을@ 줍@니다@.@@@@

**📝 정리** — `@`로 조각을 구분해 보니 토큰 단위로 왔다. 그런데 내용은 또 "Low-Cost Energy Loss"라는 지어낸 약자였다.

In [23]:
result = stream_chain.invoke({"topic": "LCEL stream"})
print(result)

LCEL(stream)은 "Low-Cost Energy Loss"의 약자로, 에너지 손실을 최소화하기 위한 기술이나 방법을 의미합니다. 주로 전력 시스템이나 에너지 관리 분야에서 사용되며, 에너지를 효율적으로 사용하고 낭비를 줄이는 데 초점을 맞추고 있습니다.

간단히 말해, LCEL stream은 에너지를 더 저렴하고 효율적으로 사용하는 방법을 찾는 과정이라고 생각하면 됩니다. 이를 통해 에너지 비용을 절감하고 환경에도 긍정적인 영향을 미칠 수 있습니다.


### 10. 배치 실행 예제

`batch()`는 여러 입력을 한 번에 처리할 때 사용한다.

LLM API를 여러 번 호출해야 하는 경우, 반복문보다 깔끔하게 작성할 수 있다.

In [24]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

explain_chain = (
    ChatPromptTemplate.from_template("{keyword}를 한 문장으로 쉽게 설명해줘.")
    | model
    | StrOutputParser()
)

batch_inputs = [
    {"keyword": "LCEL"},
    {"keyword": "Runnable"},
    {"keyword": "RAG"},
]

batch_results = explain_chain.batch(batch_inputs)

print(type(batch_results))

for keyword, result in zip(batch_inputs, batch_results):
    print(f"[{keyword['keyword']}]")
    print(result)
    print()

<class 'list'>
[LCEL]
LCEL은 "Learning Content Exchange Language"의 약자로, 학습 콘텐츠를 서로 교환하고 공유하기 위한 표준 형식입니다.

[Runnable]
Runnable은 자바에서 스레드를 실행할 수 있는 작업을 정의하는 인터페이스로, run() 메서드를 구현하여 실행할 코드를 작성하는 데 사용됩니다.

[RAG]
RAG는 "Retrieval-Augmented Generation"의 약자로, 정보를 검색하여 그 결과를 바탕으로 새로운 내용을 생성하는 AI 기술입니다.



**📝 정리** — batch 결과: LCEL은 "Learning Content Exchange Language"(❌), Runnable은 **자바의 Runnable 인터페이스**(맥락 없이 단어만 주면 더 흔한 뜻으로 답함), RAG만 맞았다.

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

### 11. LCEL 기반 간단한 문서 요약기

입력으로 문서를 넣으면:

1. 문서 길이를 계산하고
2. 미리보기를 만들고
3. LLM에게 요약을 요청한다.

In [25]:
summary_preprocess_chain = RunnablePassthrough.assign(
    length=lambda x: len(x["document"]),
    preview=lambda x: x["document"][:80] + "..." if len(x["document"]) > 80 else x["document"],
)

summary_prompt = ChatPromptTemplate.from_template("""
다음 문서를 요약해줘.

문서 길이: {length}
문서 미리보기: {preview}

[문서]
{document}

요구사항:
- 핵심 내용 3개 bullet로 정리
- 마지막에 한 줄 결론 추가
""")

sample_document = """
LCEL은 LangChain Expression Language의 약자이며, LangChain에서 체인을 선언적으로 구성하기 위한 방식이다.
PromptTemplate, ChatModel, OutputParser, Retriever, RunnableLambda 같은 구성 요소를 파이프 연산자(|)로 연결할 수 있다.
LCEL을 사용하면 invoke, batch, stream 같은 공통 실행 인터페이스를 통해 단일 실행, 배치 실행, 스트리밍 실행을 일관되게 다룰 수 있다.
또한 RunnablePassthrough, RunnableParallel, RunnableBranch 등을 활용하면 입력 유지, 병렬 구성, 조건 분기 같은 패턴도 표현할 수 있다.
"""
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

summary_chain = summary_preprocess_chain | summary_prompt | model | StrOutputParser()
                        # 전처리 체인(입력 준비 단계) -> 프롬프트 -> 모델 -> 출력 파서

summary = summary_chain.invoke({"document": sample_document})
print(summary)


- LCEL(언어 체인 표현 언어)은 LangChain에서 체인을 선언적으로 구성하는 방법이다.
- 구성 요소들은 파이프 연산자(|)로 연결되어, invoke, batch, stream과 같은 실행 인터페이스를 통해 다양한 실행 방식을 지원한다.
- RunnablePassthrough, RunnableParallel, RunnableBranch 등을 사용하여 입력 유지, 병렬 처리, 조건 분기 등의 패턴을 표현할 수 있다.

결론: LCEL은 LangChain의 유연한 체인 구성과 실행 방식을 제공하는 강력한 도구이다.


### 12. LCEL 핵심 패턴 요약

자주 쓰는 패턴만 다시 정리하면 다음과 같다.

#### 1. 기본 LLM 체인

```python
chain = prompt | model | parser
```

#### 2. 전처리 후 LLM 호출

```python
chain = preprocess | prompt | model | parser
```

#### 3. RAG

```python
chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | parser
)
```

#### 4. 기존 입력에 값 추가

```python
chain = RunnablePassthrough.assign(
    new_key=lambda x: ...
)
```

#### 5. 조건 분기

```python
chain = RunnableBranch(
    (condition_a, chain_a),
    (condition_b, chain_b),
    default_chain,
)
```

LCEL을 공부할 때는 항상 다음 질문을 하면 된다.

```text
이 단계의 입력 shape은 무엇인가?
이 단계의 출력 shape은 무엇인가?
다음 단계가 그 출력을 받을 수 있는가?
```

이 감각이 잡히면 LangChain 코드를 훨씬 편하게 읽고 수정할 수 있다.

### 응용: Runnable을 반환하는 동적 라우팅

아래 함수는 **문자열 답변이 아니라 실행할 체인 자체(Runnable)** 반환합니다. `RunnableLambda`는 반환된 체인을 입력 데이터에 적용합니다. Ollama 키워드는 Ollama 체인으로 연결하고, 나머지는 OpenAI 체인으로 연결합니다. 키워드로 모델을 고르는 데모일 뿐 질문 의미를 이해하는 지능형 라우팅은 아닙니다.

In [26]:
# 미니 실습: 질문에 포함된 키워드에 따라 OpenAI / Ollama 선택
# 이 셀을 실행하면 아래 두 개 셀에서 full_chain을 호출할 수 있습니다.
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

route_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다."),
    ("human", "{question}"),
])

gpt_chain = route_prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0.7) | StrOutputParser()
ollama_chain = route_prompt | ChatOllama(model="exaone3.5", temperature=0.7) | StrOutputParser()

def route_by_keyword(inputs: dict):
    question = inputs["question"].lower()
    if "ollama" in question:
        print("선택한 모델: Ollama (exaone3.5)")
        return ollama_chain
    print("선택한 모델: OpenAI (gpt-4o-mini)")
    return gpt_chain

In [27]:
# RunnableLambda에서 체인을 반환하면 LangChain이 반환된 Runnable을 이어서 실행합니다.
full_chain = RunnableLambda(route_by_keyword)

In [28]:
q1 = "ollama로 실행 중인 exaone3.5 모델의 특징을 설명해줘."
print(f"질문: {q1}")
response1 = full_chain.invoke({"question": q1})
print(f"답변:\n{response1}")

질문: ollama로 실행 중인 exaone3.5 모델의 특징을 설명해줘.
선택한 모델: Ollama (exaone3.5)
답변:
죄송합니다만, "ollama"라는 이름의 특정 실행 환경이나 "exaone3.5" 모델에 대한 정보를 제 데이터베이스에서 직접적으로 확인할 수는 없습니다. 특히, "ollama"가 특정 플랫폼이나 소프트웨어를 가리키는지 명확하지 않아 자세한 특징을 설명하기 어렵습니다.

하지만 일반적으로 **EXAONE 3.5** 모델에 대해 말씀드리자면:

* **개발**: LG AI Research에서 개발된 대규모 언어 모델입니다.
* **크기**:  정확한 모델 크기(예: 매개변수 수)는 공개되지 않았지만, 이전 버전인 EXAONE에 비해 향상된 성능을 갖추고 있다고 알려져 있습니다.
* **목적**: 다양한 자연어 처리 작업에 최적화되어 있으며, 텍스트 생성, 이해, 요약 등에 강점을 보입니다.
* **오픈 소스**:  일부 제한 사항이 있지만, 연구 및 개발 목적으로 사용 가능한 오픈 소스 모델로 공개되었습니다.

**"ollama"** 관련 정보가 더 필요하시다면, 해당 플랫폼이나 서비스에 대한 구체적인 맥락이나 출처를 제공해 주시면 더 정확한 도움을 드릴 수 있을 것 같습니다.


**📝 정리** — "ollama"라는 키워드로 EXAONE 체인이 선택됐는데, EXAONE이 **자기 자신(exaone3.5)과 ollama를 모른다**고 답했다. 키워드 라우팅은 의미를 이해하지 못하고, 모델은 자기 자신에 대한 정보도 모를 수 있다.

In [29]:
# input()은 실행할 때 직접 질문을 입력받는다 (저장된 실행 결과에서는 DALL-E에 대한 질문이 들어갔다)
q2 = input("질문을 입력하세요 :")

In [30]:
response2 = full_chain.invoke({"question": q2})
print(f"답변:\n{response2}")

선택한 모델: OpenAI (gpt-4o-mini)
답변:
OpenAI에서 출시한 DALL-E는 이미지 생성 모델로, 사용자가 제공한 텍스트 설명을 바탕으로 독창적인 이미지를 생성하는 능력을 가지고 있습니다. DALL-E는 "DALL-E"라는 이름이 "Salvador Dalí"와 "WALL-E"라는 두 단어의 조합으로, 예술적이고 창의적인 이미지를 생성하는 데 중점을 둡니다.

DALL-E는 다양한 스타일과 주제를 처리할 수 있으며, 사용자가 요구하는 특정 세부 사항을 반영하여 이미지를 생성할 수 있습니다. 예를 들어, "우주를 배경으로 한 고양이"라는 설명을 입력하면, 해당 설명에 맞는 독창적인 이미지를 만들어냅니다.

DALL-E의 핵심 기술은 생성적 적대 신경망(GAN)과 변형 기반 모델을 사용하여 학습된 데이터에서 패턴을 인식하고, 그 패턴을 바탕으로 새로운 이미지를 생성하는 것입니다. 이 모델은 예술, 디자인, 광고 등 다양한 분야에서 활용될 수 있습니다.

OpenAI는 DALL-E를 통해 창의적인 작업을 지원하고, 사용자에게 새로운 시각적 표현의 가능성을 열어주고자 합니다. 모델은 지속적으로 개선되고 있으며, 사용자 피드백을 기반으로 발전해 나가고 있습니다.


> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

## 🔁 복습 질문

<details>
<summary>1. <code>chain.batch([...])</code>의 결과 타입은?</summary>

입력 순서대로 결과가 담긴 `list`.
</details>

<details>
<summary>2. <code>RunnableParallel({"a": ..., "b": ...})</code>에 문자열 하나를 넣으면 결과는?</summary>

각 키의 Runnable이 같은 입력으로 실행되어 `{"a": 결과, "b": 결과}` dict가 된다.
</details>

<details>
<summary>3. RAG 체인에서 <code>"question": RunnablePassthrough()</code>의 역할은?</summary>

원래 질문을 그대로 프롬프트의 `{question}` 자리로 넘긴다. `context`는 검색 결과로 채워진다.
</details>

<details>
<summary>4. 세 모델이 "LCEL"을 틀리게 설명한 것에서 얻은 교훈은?</summary>

모델이 모르는 최신 용어는 그럴듯하게 지어낸다. 정확해야 하는 내용은 문서를 context로 넣어 근거 안에서 답하게 한다.
</details>
